In [1]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import geopandas as gpd
from datetime import timedelta

from data.constants import DNC_START, DNC_END, LOCAL_CRS, WORLD_CRS
from data.datemath import from_ymd, date_aggs, to_yw
from power import power_reg

In [ ]:
point_panel_in = "../data/interim/point_panel.parquet"
tract_panel_in = "../data/interim/tract_panel.parquet"
tracts_in = "../data/interim/tracts.geoparquet"

# Pipeline In

In [3]:
point_panel = gpd.read_parquet(point_panel_in)
tract_panel = gpd.read_parquet(tract_panel_in)
tracts = gpd.read_parquet(tracts_in)['geoid10']

# Aggregate over Time

In [ ]:
feature_cols = ['uc_400', 'uc_800', 'uc_1600', 
                'mp_400', 'mp_800', 'mp_1600',
                'TOTAL_POPULATION', 'MEDIAN_HOUSEHOLD_INCOME',
                'distance_loop', 'distance_ohare', 'distance_midway', 
                'distance_uc', 'distance_mp',
                'landarea', 'train_distance', 'bike_distance', 'bus_distance',
                'train_contained', 'bike_contained', 'bus_contained']

In [11]:
def model_agg(df):
    unit_aggs = {k: 'first' for k in ['UCMP','airport','lat','long','landarea',
                                      'train_distance', 'train_contained', 
                                      'bus_distance', 'bus_contained', 
                                       'bike_distance', 'bike_contained',
                                        'uber_distance', 'uber_contained']}
    time_aggs = {'rides':'sum','DNC':'max', 'is_weekend': 'max', 'dotw': 'max'}  
    aggs = {k:v for k,v in (unit_aggs | time_aggs).items() if k in df.columns}
    # weekend is constant for weekly model but we just won't include it in the model spec
    df_weekly = df.groupby(['year-week','id','transit'], as_index=False).agg(aggs)
    df_daily = df.groupby(['date','id','transit'], as_index=False).agg(aggs)
    return df_weekly, df_daily

point_panel_weekly, point_panel_daily = model_agg(point_panel)
tract_panel_weekly, tract_panel_daily = model_agg(tract_panel)

# Choose and Justify Agg

Monthly: obviously stupid. attenuates treatment too much.

Weekly: we have 4/7 days of the week in the DNC. and no weekend days actually.
so we attenuate our signal by 3/7. 

Daily: we have more trouble modeling dotw variation. (maybe not an issue after eliminating weekends)

## with non-treatment model

In [12]:
def partial_r2(full, reduced, full_name, reduced_name):
    better = max(full.rsquared, reduced.rsquared)
    worse = min(full.rsquared, reduced.rsquared)
    partial = (better - worse) / (1 - worse)  # Proportion of unexplained variance explained by better model.
    better_name = full_name if full.rsquared > reduced.rsquared else reduced_name
    print("better model = {}, R2 = {:.3f}, partial R2 = {:.3f}".format(better_name, better, partial))

In [13]:
# Compute the partial R2 of weekly vs daily. Only include non-treatment regressors.
formula = "np.log1p(rides) ~ transit + (lat + long)**2 + I(lat**2) + I(long**2)"
full = sm.OLS.from_formula(formula, point_panel_daily).fit()
reduced = sm.OLS.from_formula(formula, point_panel_weekly).fit()
partial_r2(full, reduced, "daily", "weekly")
# The weekly model actually has a slightly higher R2, 
# meaning we're not great at explaining DOTW variation.

better model = daily, R2 = 0.771, partial R2 = 0.072


In [14]:
# Power analysis:
DNC_ATTEND = 5e4
effect_sum = 4 * 2 * DNC_ATTEND  # days in event * rides per day * attendees

pd.concat([power_reg(full, np.log1p(effect_sum/len(point_panel_daily))).rename('daily'),
           power_reg(reduced, np.log1p(effect_sum/len(point_panel_weekly))).rename('weekly')],axis=1) \
        .assign(ratio=lambda x: x['weekly']/x['daily'])
# The daily model has smaller residual variance but identical power achieved.

,daily,weekly,ratio
mean,3.440307,4.203192,1.221749
std,1.025159,1.325126,1.292605
mde,0.009888,0.02897,2.929808
effect_size,1.747667,3.233077,1.849939
power_achieved,1.0,1.0,1.0
t_stat,495.169209,312.659705,0.63142
is_detectable,True,True,1.0


## with treatment model

In [15]:
# Compute the partial R2 of weekly vs daily. 
formula = "np.log1p(rides) ~ DNC * transit + (lat + long)**2 + I(lat**2) + I(long**2)"
full = sm.OLS.from_formula(formula, point_panel_daily).fit()
reduced = sm.OLS.from_formula(formula, point_panel_weekly).fit()
partial_r2(full, reduced, "daily", "weekly")
# The weekly model actually has a slightly higher R2, 
# meaning we're not great at explaining DOTW variation.

better model = daily, R2 = 0.771, partial R2 = 0.072


In [16]:
pd.concat([power_reg(full, np.log1p(effect_sum/len(point_panel_daily)), "DNC").rename('daily'),
           power_reg(reduced, np.log1p(effect_sum/len(point_panel_weekly)), "DNC").rename('weekly')],axis=1) 
# The standard errors are smaller on the daily model but achieved effect size is net the same,
# probably because we didn't change the effect delta but scaled down the nobs and effect size
# by the same amount.

,daily,weekly
mean,0.0,0.0
std,0.018315,0.042387
mde,0.000177,0.000927
effect_size,1.747667,3.233077
power_achieved,1.0,1.0
t_stat,27716.201418,9774.532814
is_detectable,True,True


I think we'll keep the daily observations because it gives us more flexibility to 
switch to alternative treatment specifications (buffer or event study) later on.